In [4]:
import pandas as pd
from pathlib import Path

cwd = Path.cwd()
data_path = next(
    (parent / "data/raw/heart_disease.csv"
     for parent in [cwd, *cwd.parents]
     if (parent / "data/raw/heart_disease.csv").exists()),
    None,
)

if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/heart_disease.csv")

df = pd.read_csv(data_path)
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [8]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
Using cached scikit_learn-1.9.1-cp313-cp313-macosx_12_0_arm64.whl (8.3 MB)
Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl (20.5 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [9]:
import sys, json
from pathlib import Path
import pandas as pd, numpy as np

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "data/raw/heart_disease.csv").exists())
sys.path.insert(0, str(REPO_ROOT))
ARTIFACTS = REPO_ROOT / "training" / "artifacts"

from training.src.features.preprocessing import load_xy, build_preprocessor

df = pd.read_csv(REPO_ROOT / "data/raw/heart_disease.csv")
X, y = load_xy(df)
print(X.shape, y.value_counts().to_dict())

(302, 13) {1: 164, 0: 138}


In [10]:
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
prep = build_preprocessor().fit(X_tr)          # fit on TRAIN only
Z_tr, Z_va = prep.transform(X_tr), prep.transform(X_va)   # validation only transformed
print("train:", Z_tr.shape, "val:", Z_va.shape)

scaler = prep.named_steps["columns"].named_transformers_["num"].named_steps["scale"]
print("scaler means come from train only:", scaler.mean_.round(2))

train: (241, 29) val: (61, 29)
scaler means come from train only: [ 54.34 131.45 244.3  150.07   1.07   0.91]


In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.pipeline import Pipeline

SEED = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate_cv(model, name, threshold=0.5):
    rows = []
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])   # preprocessing re-fit every fold
        proba = m.predict_proba(X.iloc[va])[:, 1]
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y.iloc[va], pred, labels=[0, 1]).ravel()
        rows.append(dict(model=name, fold=fold,
                         sensitivity=tp / (tp + fn), specificity=tn / (tn + fp),
                         f1=f1_score(y.iloc[va], pred), roc_auc=roc_auc_score(y.iloc[va], proba)))
    return pd.DataFrame(rows)

dummy  = Pipeline([("prep", build_preprocessor()), ("clf", DummyClassifier(strategy="most_frequent"))])
logreg = Pipeline([("prep", build_preprocessor()), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])

results = pd.concat([evaluate_cv(dummy, "dummy"), evaluate_cv(logreg, "logreg")])
summary = results.groupby("model")[["sensitivity", "specificity", "f1", "roc_auc"]].agg(["mean", "std"]).round(3)
summary

sensitivity        specificity            f1        roc_auc       
              mean    std        mean    std   mean    std    mean    std
model                                                                    
dummy        1.000  0.000       0.000  0.000  0.704  0.006   0.500  0.000
logreg       0.915  0.058       0.803  0.148  0.881  0.047   0.917  0.054

In [12]:
results.to_csv(ARTIFACTS / "baseline_cv_folds.csv", index=False)
(ARTIFACTS / "baseline_cv_summary.json").write_text(
    json.dumps(json.loads(summary.to_json()), indent=2))

568